# 01 — Data Ingestion

**Goal:** Download 6 years of OHLCV data for AAPL, JPM, XOM, AMZN, SPY, GLD, compute return features, attach VIX, and persist a clean long-format CSV.

We also use **DuckDB** — an in-process analytical SQL engine — to validate and explore the ingested data with SQL queries directly on the DataFrame. This avoids any external database server while still letting us write expressive SQL in the notebook.

In [1]:
# ── 0. Install dependencies ────────────────────────────────────────────────
import subprocess, sys

packages = [
    'yfinance', 'pandas-ta', 'prophet', 'plotly',
    'scikit-learn', 'kaleido', 'duckdb', 'seaborn', 'nbformat'
]
for pkg in packages:
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f'  ✓ {pkg}')
        else:
            print(f'  ✗ {pkg}: {result.stderr[:120]}')
    except Exception as e:
        print(f'  ✗ {pkg}: {e}')
print('\nAll installs attempted.')

  ✓ yfinance


  ✗ pandas-ta:   error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1



  ✓ prophet


  ✓ plotly


  ✓ scikit-learn


  ✓ kaleido


  ✓ duckdb


  ✓ seaborn


  ✓ nbformat

All installs attempted.


In [2]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import warnings, os, math
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
import duckdb
from pathlib import Path

print('pandas', pd.__version__)
print('numpy ', np.__version__)
import yfinance; print('yfinance', yfinance.__version__)
import duckdb; print('duckdb  ', duckdb.__version__)

pandas 3.0.1
numpy  2.4.2
yfinance 1.2.0
duckdb   1.5.1


## Configuration

We define tickers, date range, and output paths in one place so all notebooks share the same convention.

In [3]:
# ── 2. Config ──────────────────────────────────────────────────────────────
TICKERS   = ['AAPL', 'JPM', 'XOM', 'AMZN', 'SPY', 'GLD']
START     = '2018-01-01'
END       = pd.Timestamp.today().strftime('%Y-%m-%d')
DATA_DIR  = Path('data')
OUT_DIR   = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

OUT_CSV   = DATA_DIR / 'market_data.csv'
SRC_CSV   = Path('market_data.csv')   # pre-supplied root-level file

print(f'Tickers : {TICKERS}')
print(f'Period  : {START}  →  {END}')
print(f'Output  : {OUT_CSV}')

Tickers : ['AAPL', 'JPM', 'XOM', 'AMZN', 'SPY', 'GLD']
Period  : 2018-01-01  →  2026-03-31
Output  : data\market_data.csv


## Download via yfinance (with fallback)

`yfinance` wraps Yahoo Finance's unofficial API. We download adjusted-close OHLCV data for all tickers in one batch call, then reshape the multi-level column index into a clean long-format table.

If the download fails (no internet, rate-limit, etc.) we fall back to the pre-supplied `market_data.csv` at the project root.

In [4]:
# ── 3. Download OHLCV ──────────────────────────────────────────────────────
def download_ohlcv(tickers, start, end):
    """Download adjusted OHLCV from yfinance, return wide DataFrame."""
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    # raw has MultiIndex columns: (field, ticker)
    raw.columns.names = ['Field', 'Ticker']
    return raw

try:
    print('Attempting yfinance download...')
    raw = download_ohlcv(TICKERS, START, END)
    print(f'  Downloaded: {raw.shape[0]} trading days × {raw.shape[1]} columns')
    USE_FALLBACK = False
except Exception as e:
    print(f'  yfinance failed: {e}')
    USE_FALLBACK = True

Attempting yfinance download...


  Downloaded: 2071 trading days × 30 columns


In [5]:
# ── 4. Reshape wide → long ─────────────────────────────────────────────────
# We want one row per (Date, Ticker) with columns: Open, High, Low, Close, Volume

if not USE_FALLBACK:
    frames = []
    for ticker in TICKERS:
        try:
            sub = raw.xs(ticker, axis=1, level='Ticker')[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
            sub = sub.dropna(subset=['Close'])
            sub['Ticker'] = ticker
            sub.index.name = 'Date'
            sub = sub.reset_index()
            frames.append(sub)
        except KeyError:
            print(f'  WARNING: {ticker} not found in download — skipping')

    df = pd.concat(frames, ignore_index=True)
    df['Date'] = pd.to_datetime(df['Date']).dt.normalize()  # strip time component
    df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
    print(f'Long-format shape: {df.shape}')
    print(df.head(3))

Long-format shape: (12426, 7)
Field       Date       Open       High        Low      Close     Volume Ticker
0     2018-01-02  39.812816  40.313518  39.602239  40.304157  102223600   AAPL
1     2018-01-03  40.367350  40.839976  40.233987  40.297157  118071600   AAPL
2     2018-01-04  40.369681  40.587278  40.262056  40.484329   89738400   AAPL


## Feature Engineering — Returns

Three return measures capture different aspects of price movement:

| Feature | Formula | Why |
|---|---|---|
| **DailyReturn** | `pct_change()` | Arithmetic return — intuitive, used in ML features |
| **LogReturn** | `log(Pₜ / Pₜ₋₁)` | Additive over time, symmetric, preferred in statistics |
| **CumReturn** | `(1+r).cumprod() − 1` | Total return from inception — used for rebasing charts |

In [6]:
# ── 5. Compute return features ─────────────────────────────────────────────
if not USE_FALLBACK:
    df = df.sort_values(['Ticker', 'Date'])

    df['DailyReturn'] = df.groupby('Ticker')['Close'].pct_change()
    df['LogReturn']   = df.groupby('Ticker')['Close'].transform(
                            lambda x: np.log(x / x.shift(1)))
    df['CumReturn']   = df.groupby('Ticker')['DailyReturn'].transform(
                            lambda x: (1 + x).cumprod() - 1)

    print('Return feature nulls:')
    print(df[['DailyReturn','LogReturn','CumReturn']].isnull().sum())

Return feature nulls:
Field
DailyReturn    6
LogReturn      6
CumReturn      6
dtype: int64


## Add VIX (Fear Index)

VIX (CBOE Volatility Index) measures implied 30-day volatility of the S&P 500. Merging it into every row lets the anomaly model in notebook 04 use market-wide fear as a feature.

In [7]:
# ── 6. Download VIX and merge ──────────────────────────────────────────────
if not USE_FALLBACK:
    try:
        vix_raw = yf.download('^VIX', start=START, end=END, auto_adjust=True, progress=False)
        vix = vix_raw[['Close']].rename(columns={'Close': '^VIX'})
        vix.index.name = 'Date'
        vix = vix.reset_index()
        vix['Date'] = pd.to_datetime(vix['Date']).dt.normalize()
        # Remove any multi-level columns if present
        if hasattr(vix.columns, 'levels'):
            vix.columns = ['Date', '^VIX']
        df = df.merge(vix, on='Date', how='left')
        df['^VIX'] = df['^VIX'].ffill()   # fill weekends/holidays
        print(f'VIX merged. Nulls in ^VIX: {df["^VIX"].isnull().sum()}')
    except Exception as e:
        print(f'VIX download failed: {e} — setting ^VIX = NaN')
        df['^VIX'] = np.nan

VIX merged. Nulls in ^VIX: 0


## Fallback — use pre-supplied CSV

If yfinance was unavailable, we load the bundled `market_data.csv` (already in the correct long format with all features pre-computed) and copy it to `data/market_data.csv`.

In [8]:
# ── 7. Fallback to pre-supplied CSV ────────────────────────────────────────
if USE_FALLBACK:
    if SRC_CSV.exists():
        df = pd.read_csv(SRC_CSV, parse_dates=['Date'])
        df['Date'] = pd.to_datetime(df['Date']).dt.normalize()
        df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
        print(f'Loaded fallback CSV: {df.shape}')
    else:
        raise FileNotFoundError(
            'yfinance download failed AND market_data.csv not found. '
            'Please check your internet connection or place market_data.csv '
            'in the project root.')

# Ensure correct column order regardless of source
expected_cols = ['Date', 'Ticker', 'Close', 'Volume', 'High', 'Low',
                 'DailyReturn', 'LogReturn', 'CumReturn', '^VIX']
existing_cols = [c for c in expected_cols if c in df.columns]
df = df[existing_cols]
print('Final columns:', df.columns.tolist())

Final columns: ['Date', 'Ticker', 'Close', 'Volume', 'High', 'Low', 'DailyReturn', 'LogReturn', 'CumReturn', '^VIX']


In [9]:
# ── 8. Save to data/market_data.csv ───────────────────────────────────────
df.to_csv(OUT_CSV, index=False)
print(f'Saved → {OUT_CSV}')

Saved → data\market_data.csv


## SQL Exploration with DuckDB

DuckDB can run SQL queries **directly on a pandas DataFrame** — no server, no import step. This is extremely useful for quick aggregations and sanity checks that would be verbose in pandas.

In [10]:
# ── 9. DuckDB — ticker summary ─────────────────────────────────────────────
con = duckdb.connect()
con.register('market', df)

ticker_summary = con.execute("""
    SELECT
        Ticker,
        COUNT(*)                                    AS trading_days,
        MIN(Date)                                   AS first_date,
        MAX(Date)                                   AS last_date,
        ROUND(MIN(Close), 2)                        AS price_min,
        ROUND(MAX(Close), 2)                        AS price_max,
        ROUND(AVG(DailyReturn) * 252 * 100, 2)      AS ann_return_pct,
        ROUND(STDDEV(DailyReturn) * SQRT(252) * 100, 2) AS ann_vol_pct,
        SUM(CASE WHEN DailyReturn < -0.03 THEN 1 ELSE 0 END) AS days_down_3pct
    FROM market
    GROUP BY Ticker
    ORDER BY ann_return_pct DESC
""").df()

print('=== Ticker Summary (SQL via DuckDB) ===')
print(ticker_summary.to_string(index=False))

=== Ticker Summary (SQL via DuckDB) ===
Ticker  trading_days first_date  last_date  price_min  price_max  ann_return_pct  ann_vol_pct  days_down_3pct
  AAPL          2071 2018-01-02 2026-03-30      33.77     285.92           26.74        30.63           101.0
  AMZN          2071 2018-01-02 2026-03-30      59.45     254.00           20.71        34.32           129.0
   JPM          2071 2018-01-02 2026-03-30      67.10     334.61           18.67        28.94            76.0
   XOM          2071 2018-01-02 2026-03-30      23.99     171.47           17.61        30.10            91.0
   GLD          2071 2018-01-02 2026-03-30     111.10     495.90           15.95        16.49            22.0
   SPY          2071 2018-01-02 2026-03-30     204.94     693.60           13.83        19.31            32.0


In [11]:
# ── 10. DuckDB — top 10 worst daily drops across all tickers ──────────────
worst_days = con.execute("""
    SELECT
        Date,
        Ticker,
        ROUND(Close, 2)         AS Close,
        ROUND(DailyReturn * 100, 2) AS DailyReturn_Pct,
        ROUND("^VIX", 1)        AS VIX
    FROM market
    WHERE DailyReturn IS NOT NULL
    ORDER BY DailyReturn ASC
    LIMIT 10
""").df()

print('=== Top 10 Worst Daily Drops ===')
print(worst_days.to_string(index=False))

=== Top 10 Worst Daily Drops ===
      Date Ticker  Close  DailyReturn_Pct  VIX
2020-03-16    JPM  75.03           -14.96 82.7
2022-04-29   AMZN 124.28           -14.05 33.4
2020-03-09    JPM  79.34           -13.55 54.5
2020-03-16   AAPL  58.52           -12.86 82.7
2020-03-09    XOM  31.92           -12.22 54.5
2020-03-12    XOM  28.36           -11.43 75.5
2020-03-16    SPY 219.19           -10.94 82.7
2020-03-18    JPM  71.23           -10.53 76.4
2026-01-30    GLD 444.95           -10.27 17.4
2020-03-18    XOM  25.26           -10.02 76.4


In [12]:
# ── 11. Final dataset validation ──────────────────────────────────────────
print('='*55)
print(f'Shape          : {df.shape}')
print(f'Date range     : {df.Date.min().date()}  →  {df.Date.max().date()}')
print(f'Tickers        : {sorted(df.Ticker.unique().tolist())}')
print(f'Null counts    :')
print(df.isnull().sum().to_string())
print('='*55)
print('\nSample rows:')
print(df.head(6).to_string(index=False))

Shape          : (12426, 10)
Date range     : 2018-01-02  →  2026-03-30
Tickers        : ['AAPL', 'AMZN', 'GLD', 'JPM', 'SPY', 'XOM']
Null counts    :
Date           0
Ticker         0
Close          0
Volume         0
High           0
Low            0
DailyReturn    6
LogReturn      6
CumReturn      6
^VIX           0

Sample rows:
      Date Ticker     Close    Volume      High       Low  DailyReturn  LogReturn  CumReturn  ^VIX
2018-01-02   AAPL 40.304157 102223600 40.313518 39.602239          NaN        NaN        NaN  9.77
2018-01-03   AAPL 40.297157 118071600 40.839976 40.233987    -0.000174  -0.000174  -0.000174  9.15
2018-01-04   AAPL 40.484329  89738400 40.587278 40.262056     0.004645   0.004634   0.004470  9.22
2018-01-05   AAPL 40.945263  94640000 41.031832 40.489016     0.011385   0.011321   0.015907  9.22
2018-01-08   AAPL 40.793194  82271200 41.087999 40.694922    -0.003714  -0.003721   0.012134  9.52
2018-01-09   AAPL 40.788506  86336000 40.959305 40.573251    -0.000115 

In [13]:
print('\n✓ 01_ingestion complete')
print(f'  → {OUT_CSV} ({OUT_CSV.stat().st_size / 1024:.1f} KB)')


✓ 01_ingestion complete
  → data\market_data.csv (1943.3 KB)
